# 37 — Operating Regimes v2

**Engineering statement:** Operating regimes specify adaptive confidence scheduling.

Notebook 37 v2 is the synthesis notebook for the first systems arc of the repository. It turns confidence profiles, throughput objectives, and hardware feasibility constraints into an adaptive regime-selection problem.

Notebook 29 asked which schedules are feasible. Notebook 37 asks which policy should be active once the serving system enters a specific operating regime.


## Repository roadmap

```text
00 Context
07 Verification Resource
13 Confidence Scheduling
17 Semi-Autoregressive Design
23 Throughput Objective
29 Hardware-Aware Scheduling
37 Operating Regimes
43 Resource Allocation
49 Adaptive AI Infrastructure
```


## Start here

```text
confidence profiles
→ throughput objective
→ hardware constraints
→ feasible schedules
→ operating regimes
→ adaptive serving policy
→ closed-loop controller
```

A fixed schedule is a local rule. An operating-regime policy is a deployment rule.

Notebook 37 v2 emphasizes:

1. **smooth regime boundaries**, not hard lookup boxes;
2. **load-dependent optima**, not a single best prefix length;
3. **nonlinear regime transitions**, not a one-way chain;
4. **closed-loop feedback**, not static scheduling.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import FileLink, display

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "37_operating_regimes"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=12):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.4,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end, style="->", lw=1.8):
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle=style, lw=lw))


## 1. Operating-regime map

A regime is a region of serving-state space where one scheduling behavior remains appropriate.

The center is throughput-balanced. The four primary directions are:

- confidence-bound,
- compute-bound,
- memory-bound,
- latency-bound.

The corners are mixed regimes, where two constraints jointly shape the scheduling policy.


In [ ]:
# 37_operating_regime_map.png

fig, ax = plt.subplots(figsize=(8.2, 7))
ax.axis("off")
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)

ax.annotate("", xy=(3.5, 0), xytext=(-3.5, 0), arrowprops=dict(arrowstyle="<->", lw=2))
ax.annotate("", xy=(0, 3.5), xytext=(0, -3.5), arrowprops=dict(arrowstyle="<->", lw=2))

ax.text(0, 3.65, "Memory bound", ha="center", va="bottom", fontsize=15, fontweight="bold")
ax.text(0, -3.65, "Latency bound", ha="center", va="top", fontsize=15, fontweight="bold")
ax.text(-3.65, 0, "Compute\nbound", ha="right", va="center", fontsize=15, fontweight="bold")
ax.text(3.65, 0, "Confidence\nbound", ha="left", va="center", fontsize=15, fontweight="bold")

rounded_box(ax, (-1.15, -0.45), 2.3, 0.9, "Throughput-\nbalanced", fontsize=13)
rounded_box(ax, (-3.2, 2.1), 1.7, 0.75, "memory +\ncompute", fontsize=11)
rounded_box(ax, (1.5, 2.1), 1.7, 0.75, "memory +\nconfidence", fontsize=11)
rounded_box(ax, (-3.2, -2.9), 1.7, 0.75, "latency +\ncompute", fontsize=11)
rounded_box(ax, (1.5, -2.9), 1.7, 0.75, "latency +\nconfidence", fontsize=11)

ax.set_title("Operating regimes specify adaptive confidence scheduling", fontsize=20, pad=20)
savefig("37_operating_regime_map.png")


## 2. Smooth phase diagram

The first version used rectangular regions. v2 uses smooth decision boundaries to better represent an adaptive controller.

The axes remain:

- \(p\): draft predictability / confidence floor,
- \(R\): active serving load.

The selected regime changes smoothly as confidence improves or load increases.


In [ ]:
# 37_phase_diagram.png — smooth-ish adaptive regime map

load = np.linspace(4, 64, 140)
predictability = np.linspace(0.2, 0.9, 150)
P, R = np.meshgrid(predictability, load)

# Scores for regimes. Higher score wins.
confidence_score = 1.35*(1 - P) + 0.010*R
balanced_score = 1.10*np.exp(-((P-0.58)/0.22)**2) + 0.85*np.exp(-((R-18)/16)**2)
compute_score = 0.75*P + 0.018*R - 0.00018*(R-34)**2
memory_score = 0.95*P + 0.030*R - 1.05*np.exp(-((P-0.86)/0.13)**2)
latency_score = 1.15*P + 0.036*R - 0.45*np.exp(-((P-0.55)/0.18)**2)

scores = np.stack([confidence_score, balanced_score, compute_score, memory_score, latency_score], axis=0)
regime = np.argmax(scores, axis=0)

fig, ax = plt.subplots(figsize=(9.5, 6.5))
im = ax.imshow(
    regime,
    aspect="auto",
    origin="lower",
    extent=[predictability.min(), predictability.max(), load.min(), load.max()],
    interpolation="bilinear"
)
ax.set_xlabel("draft predictability / confidence floor", fontsize=14)
ax.set_ylabel("active requests R", fontsize=14)
ax.set_title("Operating-regime phase diagram", fontsize=20)

# Contour boundaries between regions.
ax.contour(P, R, regime, levels=[0.5, 1.5, 2.5, 3.5], linewidths=1.2, alpha=0.65)

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4])
cbar.ax.set_yticklabels(["confidence", "balanced", "compute", "memory", "latency"])
cbar.set_label("selected regime")

ax.text(0.29, 20, "confidence-\nlimited", fontsize=12, ha="center")
ax.text(0.58, 15, "balanced", fontsize=12, ha="center")
ax.text(0.64, 34, "compute-\nlimited", fontsize=12, ha="center")
ax.text(0.60, 53, "memory-\nlimited", fontsize=12, ha="center")
ax.text(0.81, 52, "latency-\nlimited", fontsize=12, ha="center")

savefig("37_phase_diagram.png")

phase_data = {
    "load": load.round(3).tolist(),
    "predictability": predictability.round(3).tolist(),
    "regime_labels": ["confidence-limited", "throughput-balanced", "compute-limited", "memory-limited", "latency-limited"],
    "revision": "v2_smooth_boundaries"
}
(RESULTS / "37_phase_diagram_metadata.json").write_text(json.dumps(phase_data, indent=2), encoding="utf-8")


## 3. Constraint activation

As load rises, constraints activate in different combinations. The active constraint vector is

\[
z_t=(C_{\rm conf}, C_{\rm compute}, C_{\rm memory}, C_{\rm latency}).
\]

The regime classifier estimates this vector and selects a scheduling policy.


In [ ]:
# 37_constraint_activation.png

loads = np.array([8, 20, 36, 52])
labels = ["low load", "moderate", "high", "saturated"]

confidence = np.array([0.65, 0.72, 0.78, 0.82])
compute = np.array([0.35, 0.55, 0.78, 0.95])
memory = np.array([0.25, 0.42, 0.72, 0.92])
latency = np.array([0.20, 0.38, 0.70, 0.98])

x = np.arange(len(labels))
width = 0.19

fig, ax = plt.subplots(figsize=(10.5, 5.8))
ax.bar(x - 1.5*width, confidence, width, label="confidence")
ax.bar(x - 0.5*width, compute, width, label="compute")
ax.bar(x + 0.5*width, memory, width, label="memory")
ax.bar(x + 1.5*width, latency, width, label="latency")
ax.axhline(0.75, linestyle="--", linewidth=1.5, label="active threshold")
ax.set_xticks(x, labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("constraint pressure", fontsize=14)
ax.set_title("Constraint activation changes the scheduling regime", fontsize=20)
ax.legend(ncol=3)
ax.grid(axis="y", alpha=0.3)
savefig("37_constraint_activation.png")


## 4. Policy switching

The scheduler is now a regime-conditioned policy:

\[
\pi_\rho(s_t)\rightarrow \ell^*_t.
\]

A regime classifier uses confidence, load, memory, and latency signals to select the policy.


In [ ]:
# 37_policy_switching.png

fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.axis("off")
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 4)

nodes = [
    ("request\nstream", 0.4, 2.2, 1.35, 0.75),
    ("state\nestimator", 2.35, 2.2, 1.45, 0.75),
    ("regime\nclassifier", 4.45, 2.2, 1.55, 0.75),
    ("policy\nselector", 6.65, 2.2, 1.4, 0.75),
    ("verification\nschedule", 8.75, 2.2, 1.65, 0.75),
    ("serving\noutcome", 10.95, 2.2, 1.25, 0.75),
]
for label, x0, y0, w, h in nodes:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=12)
for i in range(len(nodes)-1):
    _, x0, y0, w, h = nodes[i]
    _, x1, y1, w1, h1 = nodes[i+1]
    arrow(ax, (x0+w+0.05, y0+h/2), (x1-0.05, y1+h1/2))

signals = [
    ("confidence", 2.2, 0.65, 1.25, 0.55),
    ("load", 3.7, 0.65, 0.9, 0.55),
    ("memory", 4.85, 0.65, 1.0, 0.55),
    ("latency", 6.1, 0.65, 1.0, 0.55),
]
for label, x0, y0, w, h in signals:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=10)
    arrow(ax, (x0+w/2, y0+h+0.04), (5.2, 2.15))

ax.annotate("", xy=(2.95, 3.1), xytext=(11.6, 3.1), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.text(7.2, 3.28, "online feedback", ha="center", fontsize=11)

ax.set_title("Adaptive confidence scheduling selects policies by operating regime", fontsize=20, pad=12)
savefig("37_policy_switching.png")


## 5. Throughput landscape v2

The first version showed a throughput landscape, but the optimum stayed near the same \(\ell\). v2 makes the operating-regime idea explicit:

\[
R=8 \Rightarrow \ell^*\approx 7,\quad
R=20 \Rightarrow \ell^*\approx 6,\quad
R=36 \Rightarrow \ell^*\approx 5,\quad
R=52 \Rightarrow \ell^*\approx 4.
\]

As load increases, the optimum shifts toward shorter schedules.


In [ ]:
# 37_throughput_landscape.png — v2 shifting optima

ell = np.arange(1, 10)
loads = np.array([8, 20, 36, 52])
target_opt = {8: 7, 20: 6, 36: 5, 52: 4}

def landscape(R0, e):
    # Lower load supports longer useful schedules; high load penalizes them sooner.
    opt = target_opt[int(R0)]
    height = 1.62 / (1 + 0.018*R0)
    rise = 1 - np.exp(-0.85 * e)
    penalty = np.exp(-0.020 * np.maximum(e - opt, 0)**2 * (1 + R0/18))
    pre_penalty = np.exp(-0.006 * np.maximum(opt - e, 0)**2)
    return height * rise * penalty * pre_penalty

fig, ax = plt.subplots(figsize=(10.5, 6.2))
for R0 in loads:
    vals = np.array([landscape(R0, e) for e in ell])
    ax.plot(ell, vals, marker="o", label=f"R={R0}")
    opt_i = int(np.argmax(vals))
    ax.scatter([ell[opt_i]], [vals[opt_i]], s=100, zorder=4)
    ax.text(ell[opt_i]+0.08, vals[opt_i], f"$\\ell^*={ell[opt_i]}$", fontsize=11)

ax.set_xlabel("scheduled prefix length $\\ell$", fontsize=14)
ax.set_ylabel("throughput proxy", fontsize=14)
ax.set_title("Throughput landscape: optimum adapts to operating regime", fontsize=20)
ax.grid(True, alpha=0.3)
ax.legend(title="active requests")
savefig("37_throughput_landscape.png")


## 6. Regime transition graph v2

The first transition graph looked like a one-way chain. v2 treats regimes as a graph. The controller can move among regimes as confidence improves, load drops, memory pressure changes, or latency targets tighten.

This graph is not a time axis. It is a policy-state graph.


In [ ]:
# 37_regime_transition_graph.png — v2 graph, not a line

fig, ax = plt.subplots(figsize=(9.8, 6.2))
ax.axis("off")
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)

positions = {
    "balanced": (1.1, 2.65, 1.75, 0.8),
    "confidence": (4.1, 4.65, 1.85, 0.8),
    "compute": (4.1, 2.65, 1.85, 0.8),
    "memory": (7.0, 3.65, 1.75, 0.8),
    "latency": (7.0, 1.55, 1.75, 0.8),
}

labels = {
    "balanced": "Throughput-\nbalanced",
    "confidence": "Confidence-\nlimited",
    "compute": "Compute-\nlimited",
    "memory": "Memory-\nlimited",
    "latency": "Latency-\nlimited",
}
for key, (x0, y0, w, h) in positions.items():
    rounded_box(ax, (x0, y0), w, h, labels[key], fontsize=11)

def center(key):
    x0, y0, w, h = positions[key]
    return (x0 + w/2, y0 + h/2)

def edge(a, b, style="<->", rad=0.0):
    ax.annotate(
        "",
        xy=center(b),
        xytext=center(a),
        arrowprops=dict(
            arrowstyle=style,
            lw=1.7,
            shrinkA=35,
            shrinkB=35,
            connectionstyle=f"arc3,rad={rad}"
        )
    )

edge("balanced", "confidence", "<->", 0.06)
edge("balanced", "compute", "<->", 0.0)
edge("balanced", "latency", "<->", -0.08)
edge("confidence", "memory", "<->", 0.08)
edge("compute", "memory", "<->", 0.0)
edge("compute", "latency", "<->", 0.0)
edge("memory", "latency", "<->", 0.0)

ax.text(5.0, 0.55, "regime changes track confidence, load, memory pressure, and latency targets", ha="center", fontsize=11)
ax.set_title("Regime transition graph for adaptive serving", fontsize=20, pad=12)
savefig("37_regime_transition_graph.png")


## 7. Closed-loop controller

The capstone addition in v2 is a closed-loop controller figure.

Deployment is not a static schedule. It is a feedback-controlled scheduling system:

```text
observe state
→ classify regime
→ select policy
→ schedule verification
→ observe throughput
→ update state
```


In [ ]:
# 37_closed_loop_controller.png

fig, ax = plt.subplots(figsize=(11, 7))
ax.axis("off")
ax.set_xlim(0, 11)
ax.set_ylim(0, 7)

nodes = [
    ("Request\nstream", 1.0, 5.6, 1.55, 0.65),
    ("State\nestimator", 3.25, 5.6, 1.55, 0.65),
    ("Operating-regime\nclassifier", 5.55, 5.45, 1.9, 0.8),
    ("Policy\nselector", 8.05, 5.6, 1.45, 0.65),
    ("Verification\nscheduler", 7.8, 3.55, 1.9, 0.75),
    ("Serving\nsystem", 5.45, 2.0, 1.75, 0.7),
    ("Observed\nthroughput", 2.95, 2.0, 1.75, 0.7),
    ("State\nupdate", 1.0, 3.55, 1.55, 0.7),
]
for label, x0, y0, w, h in nodes:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=11)

# top line
arrow(ax, (2.57, 5.93), (3.20, 5.93))
arrow(ax, (4.82, 5.93), (5.50, 5.86))
arrow(ax, (7.47, 5.86), (8.00, 5.93))

# down to scheduler/system/outcome/update
arrow(ax, (8.78, 5.57), (8.78, 4.35))
arrow(ax, (7.75, 3.72), (6.95, 2.72))
arrow(ax, (5.40, 2.35), (4.75, 2.35))
arrow(ax, (2.95, 2.35), (2.55, 3.78))
arrow(ax, (1.78, 4.25), (1.78, 5.55))

# feedback shortcut
ax.annotate(
    "",
    xy=(3.95, 5.55),
    xytext=(3.8, 2.72),
    arrowprops=dict(arrowstyle="->", lw=1.4, connectionstyle="arc3,rad=-0.25")
)
ax.text(5.45, 0.8, "Deployment becomes a feedback-controlled scheduling system.", ha="center", fontsize=13)

ax.set_title("Closed-loop adaptive confidence-scheduled serving", fontsize=20, pad=12)
savefig("37_closed_loop_controller.png")


## 8. Repository summary

The first seven notebooks form a complete engineering path:

```text
context
→ resource
→ confidence
→ draft architecture
→ throughput
→ hardware constraints
→ operating regimes
```

Notebook 37 is the point where local mechanisms become a deployment-scale control map.


In [ ]:
# 37_repository_summary.png

fig, ax = plt.subplots(figsize=(13, 4.6))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 4)

nodes = [
    ("00\nContext", 0.4, 2.2, 1.25, 0.75),
    ("07\nVerification\nResource", 2.1, 2.2, 1.35, 0.75),
    ("13\nConfidence\nScheduling", 3.95, 2.2, 1.45, 0.75),
    ("17\nSemi-AR\nDrafting", 5.95, 2.2, 1.35, 0.75),
    ("23\nThroughput\nObjective", 7.8, 2.2, 1.45, 0.75),
    ("29\nHardware\nConstraints", 9.8, 2.2, 1.35, 0.75),
    ("37\nOperating\nRegimes", 11.55, 2.2, 1.25, 0.75),
]
for label, x0, y0, w, h in nodes:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=10)
for i in range(len(nodes)-1):
    _, x0, y0, w, h = nodes[i]
    _, x1, y1, w1, h1 = nodes[i+1]
    arrow(ax, (x0+w+0.05, y0+h/2), (x1-0.05, y1+h1/2))

rounded_box(ax, (3.6, 0.65), 6.1, 0.75, "Local mechanisms become deployment-scale operating regimes.", fontsize=13)
ax.set_title("Confidence-scheduled decoding: repository synthesis through Notebook 37", fontsize=20, pad=12)
savefig("37_repository_summary.png")


## Key equations

Prefix survival:

\[
a_j=\prod_{k=1}^{j}c_k
\]

Throughput objective:

\[
\Theta(\ell)=A(\ell)S(B)
\]

Feasible schedule set:

\[
\mathcal{F}
=
\{\ell:\;L(R,\ell)\le L_{\max},\;M(R,\ell)\le M_{\max}\}
\]

Constraint state:

\[
z_t=(C_{\rm conf}, C_{\rm compute}, C_{\rm memory}, C_{\rm latency})
\]

Regime classification:

\[
\rho_t=g(z_t)
\]

Regime-conditioned policy:

\[
\pi_{\rho_t}
:
(s_{\rm confidence},s_{\rm hardware})
\mapsto
\ell^*_t
\]

Closed-loop update:

\[
s_{t+1}=h(s_t,\ell^*_t,\Theta_t).
\]


## Engineering summary

Notebook 37 v2 completes the first systems arc.

A confidence-scheduled serving system should not use one fixed rule. It should identify the active operating regime, select the appropriate policy, and update the regime estimate from observed throughput and serving-state feedback.

> **Operating regimes specify adaptive confidence scheduling.**


In [ ]:
# Save notebook manifest

manifest = {
    "notebook": "37_operating_regimes_v2.ipynb",
    "title": "Operating Regimes v2",
    "engineering_statement": "Operating regimes specify adaptive confidence scheduling.",
    "figures": [
        "37_operating_regime_map.png",
        "37_phase_diagram.png",
        "37_constraint_activation.png",
        "37_policy_switching.png",
        "37_throughput_landscape.png",
        "37_regime_transition_graph.png",
        "37_closed_loop_controller.png",
        "37_repository_summary.png",
    ],
    "next_notebook": "43_resource_allocation.ipynb",
    "revision_notes": [
        "smooth phase diagram boundaries",
        "throughput optima shift under load",
        "transition graph rather than fixed chain",
        "closed-loop controller synthesis figure"
    ]
}
(RESULTS / "37_operating_regimes_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


## Download artifacts

Run the final cell to package Notebook 37 v2 outputs for download.

The archive includes:

- `37_operating_regimes_v2.ipynb`
- generated figures
- generated JSON outputs
- notebook manifest


In [ ]:
# Package Notebook 37 v2 artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "37_operating_regimes"
SITE = REPO_ROOT / "site"

RESULTS.mkdir(parents=True, exist_ok=True)
zip_path = RESULTS / "37_operating_regimes_v2_artifacts.zip"

notebook_candidates = [
    NOTEBOOKS / "37_operating_regimes_v2.ipynb",
    NOTEBOOKS / "37_operating_regimes.ipynb",
    Path.cwd() / "37_operating_regimes_v2.ipynb",
    Path.cwd() / "37_operating_regimes.ipynb",
]
notebook_path = next((p for p in notebook_candidates if p.exists()), None)

paths_to_include = []
if notebook_path is not None:
    paths_to_include.append(notebook_path)

for item in [FIGURES, RESULTS, SITE]:
    if item.exists():
        paths_to_include.append(item)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
